# 04 - Method C: Gradient-Boosted Regime Classifier

**Inertia v2 - Factor Regimes** - Sprint 2 (V9 spec)

Reframe regime detection as **classification**: label months as favorable when next-month return exceeds the in-sample median ($q=0.50$); learn nonlinear decision boundaries with a stack of 200 small decision trees.

**Features**: cross-factor lags + 6-month rolling volatilities (no macro \textemdash{} macro features hurt this model in OOS testing).

**Weight rule**: long-only with confidence ($w_t = \max(2P_t - 1, 0)$, range $[0, 1]$).

This is variant V9 from `scripts/explore_method_c.py` (the parallel-agent winner).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.data import build_factor_panel, FF5_FACTORS
from lib.methods import fit_predict_gbm_oos
from lib.backtest import prob_to_weight, apply_weights, static_factor_returns
from lib.evaluation import perf_stats
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1990-01-01'

In [2]:
# V9 spec: cross-factor only (no macro), q=0.50 label
panel = build_factor_panel(include_macro=False)
factor_features = [f'lag1_{f}' for f in FF5_FACTORS] + [f'vol6_{f}' for f in FF5_FACTORS]
print(f'Features ({len(factor_features)}): {factor_features}')

# Match Method B's OOS start so the comprehensive scoreboard window is consistent
OOS_START_HERE = '2000-02-01'

probs = {}
for f in FF5_FACTORS:
    print(f'Fitting GBM for {f}...')
    probs[f] = fit_predict_gbm_oos(panel, f, factor_features,
                                     oos_start=OOS_START_HERE, refit_months=12, min_train=120)
P = pd.DataFrame(probs)
print(f'\nOOS shape: {P.shape}, range {P.index.min().date()} -> {P.index.max().date()}')
P.describe().T[['mean','std','min','max']]


Features (10): ['lag1_MKT_RF', 'lag1_SMB', 'lag1_HML', 'lag1_RMW', 'lag1_CMA', 'vol6_MKT_RF', 'vol6_SMB', 'vol6_HML', 'vol6_RMW', 'vol6_CMA']
Fitting GBM for MKT_RF...


Fitting GBM for SMB...


Fitting GBM for HML...


Fitting GBM for RMW...


Fitting GBM for CMA...



OOS shape: (312, 5), range 2000-02-29 -> 2026-01-31


,mean,std,min,max
MKT_RF,0.5001,0.1776,0.0735,0.9493
SMB,0.5223,0.1773,0.1225,0.9423
HML,0.4998,0.1827,0.0562,0.8986
RMW,0.5113,0.1671,0.0731,0.9313
CMA,0.4775,0.1811,0.0950,0.9385


In [3]:
fig, axes = plt.subplots(5, 1, figsize=(FULL_COL, 7.0), sharex=True)
for ax, f in zip(axes, FF5_FACTORS):
    s = P[f]
    ax.plot(s.index, s.values, color=FACTOR_PALETTE[f], linewidth=0.9)
    ax.axhline(0.5, color=C['muted'], linewidth=0.4, linestyle='--')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(f, color=FACTOR_PALETTE[f], fontsize=10, fontweight='bold')
    ax.set_yticks([0, 0.5, 1])
axes[0].set_title('Method C: GBM P(favorable), 1990 to 2026',
                  loc='left', color=C['ink'])
yearly_xticks(axes[-1], every=5)
save_fig(fig, '11_method_c_probs_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/11_method_c_probs_timeseries.png


In [4]:
rows = {}
timed_returns = {}
for f in FF5_FACTORS:
    p = P[f].dropna()
    w = prob_to_weight(p, mode='longonly')
    timed = apply_weights(w, panel[f], cost_bps_oneway=5.0)
    static = static_factor_returns(panel[f], timed.index)
    timed_returns[f] = timed['r_net']
    s_static = perf_stats(static)
    s_timed  = perf_stats(timed['r_net'])
    rows[f] = {
        'sharpe_static': s_static['sharpe_ann'],
        'sharpe_timed':  s_timed['sharpe_ann'],
        'mean_static':   s_static['mean_ann'],
        'mean_timed':    s_timed['mean_ann'],
        'vol_static':    s_static['vol_ann'],
        'vol_timed':     s_timed['vol_ann'],
        'mdd_static':    s_static['max_drawdown'],
        'mdd_timed':     s_timed['max_drawdown'],
    }
compare = pd.DataFrame(rows).T
compare['sharpe_uplift'] = compare['sharpe_timed'] - compare['sharpe_static']
save_table(compare, '11_method_c_per_factor_compare', out_dir=TABLE_DIR)
compare

  saved: ../tables/11_method_c_per_factor_compare.{csv,md}


,sharpe_static,sharpe_timed,mean_static,mean_timed,vol_static,vol_timed,mdd_static,mdd_timed,sharpe_uplift
MKT_RF,0.4748,0.4717,0.0744,0.0199,0.1567,0.0423,-0.5411,-0.0824,-0.0031
SMB,0.1191,0.1378,0.0121,0.0044,0.1017,0.0318,-0.3552,-0.1050,0.0187
HML,0.2555,0.0976,0.0301,0.0031,0.1180,0.0319,-0.5779,-0.1696,-0.1579
RMW,0.6411,0.2888,0.0584,0.0087,0.0912,0.0300,-0.2327,-0.0737,-0.3523
CMA,0.3501,0.4767,0.0269,0.0138,0.0769,0.0290,-0.2725,-0.0644,0.1267


In [5]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.6))
x = np.arange(len(FF5_FACTORS))
w = 0.36
bars_s = ax.bar(x - w/2, compare['sharpe_static'].values, w, color=C['blue'],
                label='Static factor', edgecolor='white', linewidth=0.5)
bars_t = ax.bar(x + w/2, compare['sharpe_timed'].values, w, color=C['purple'],
                label='Method C timed', edgecolor='white', linewidth=0.5)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(FF5_FACTORS, fontsize=10)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Method C: timed vs static Sharpe, by factor (1990 to 2026)',
             loc='left', color=C['ink'])
ax.set_ylim(min(compare[['sharpe_static','sharpe_timed']].min().min(), 0) - 0.10,
            compare[['sharpe_static','sharpe_timed']].max().max() + 0.20)
bar_value_labels(ax, bars_s, fmt='{:+.2f}', offset=0.03, fontsize=8, color=C['blue'])
bar_value_labels(ax, bars_t, fmt='{:+.2f}', offset=0.03, fontsize=8, color=C['purple'])
legend_below(ax, ncol=2)
save_fig(fig, '12_method_c_sharpe_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/12_method_c_sharpe_bars.png


In [6]:
tr_df = pd.DataFrame(timed_returns).dropna(how='all')
method_c_composite = tr_df.mean(axis=1)
static_df = pd.DataFrame({f: panel[f].shift(-1).reindex(tr_df.index) for f in FF5_FACTORS}).dropna(how='all')
static_composite = static_df.mean(axis=1)
stats = pd.DataFrame({
    'static_eq_weight': perf_stats(static_composite),
    'method_c_timed':   perf_stats(method_c_composite),
}).T
save_table(stats, '12_method_c_composite_perf', out_dir=TABLE_DIR)
stats

  saved: ../tables/12_method_c_composite_perf.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
static_eq_weight,312.0000,0.0404,0.0545,0.7417,-0.2192,2.0881,-0.1508
method_c_timed,312.0000,0.0100,0.0150,0.6659,1.2014,3.6312,-0.0370


In [7]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.4))
for r, color, label, lw in [
    (static_composite, C['blue'],   'Equal-weight static FF5', 1.0),
    (method_c_composite, C['purple'], 'Method C composite (timed)', 1.4),
]:
    cum = (1 + r).cumprod()
    ax.plot(cum.index, cum.values, color=color, linewidth=lw, label=label)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Method C composite vs static FF5, 1990 to 2026',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=5)
legend_below(ax, ncol=2)
save_fig(fig, '13_method_c_composite_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/13_method_c_composite_cumret.png


In [8]:
P.to_csv(f'{TABLE_DIR}/13_method_c_probs.csv')
tr_df.to_csv(f'{TABLE_DIR}/14_method_c_timed_returns.csv')
print(f'Saved Method C outputs.')

Saved Method C outputs.
